# nb_mig_silver_20261116100000_SeedRegSub_TransformMappings_Unified_A

Create the metadata entried for the various silver unified tables for Registration Submissions. 

Note, since multiple developers are working on this simultaneously, the metadata insertion has been divided across multiple migration notebooks

In [15]:
%run "/05 - shared-libraries/nb_shared_common"

In [16]:
%run "/05 - shared-libraries/nb_shared_metadata"

In [17]:
%run "/05 - shared-libraries/nb_shared_migrations"

In [18]:
storage_account_name = "deintdlsaueaasqebla"
dataverse_container = "dataverse-asqaarms01de-unqde396ad0f22d448c94c779e588420"

In [20]:
#imports
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime, timezone
from delta.tables import DeltaTable
from mdw_shared.migrations import ensure_history_table, is_applied, record_migration

# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name, dataverse_container)
METADATA_BASE_PATH = data_lake_paths["metadata"]

VERSION_ID = "v1"

MIGRATION_ID = "20261116100000_SeedRegSub_TransformMappings_Unified_A"
NOTEBOOK_NAME = "nb_mig_silver_20261116100000_SeedRegSub_TransformMappings_Unified_A"

ensure_history_table(spark, METADATA_BASE_PATH)


In [21]:
if is_applied(spark, METADATA_BASE_PATH, MIGRATION_ID):
    print(f"Already applied: {MIGRATION_ID}")
else:
    now = datetime.now(timezone.utc)

    is_active = True
    created_utc = now
    updated_utc = now

    # =================================================================
    # PART 1: Transform Mappings
    # =================================================================

    NOTEBOOK_TO_EXECUTE_NONE = None
    JSON_CONFIG_NONE = None
    SOURCE_ID_NONE = None
    SOURCE_ENTITY_NONE = None

    # transform_entity_mappings contains the metadata entries to map ARMS to Clease
    # and cleanse to unify
    # Parameter 1 : Binding Id
    # Parameter 2 : Step Name
    # Parameter 3 : Source Id (only relevant for ARMS cleanse, for unifcation this is set to SOURCE_ID_NONE)
    # Parameter 4 : Order you want the transform to run
    # Parameter 5 : Source entity name
    # Parameter 6 : Target entity name
    # Parameter 7 : Any custom notebook to run for the transformation only if relevant
    # Parameter 8 : name of JSON mapping file to map fields and run relevant validations at a field level

    transform_entity_mappings = [

         # Unify Master
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, 1, "asqa_elicoscourse", "elicos_course", NOTEBOOK_TO_EXECUTE_NONE, "elicos_course.json"),

         # Unify Transaction
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, 1, "<cleansed_table_name>", "<unified_table_name>", NOTEBOOK_TO_EXECUTE_NONE, "<unified_table_name>.json"),
        
    ]

    insert_transform_entity_mappings(METADATA_BASE_PATH, transform_entity_mappings, VERSION_ID, spark)

    # TODO: remove before PR
    #record_migration(spark, METADATA_BASE_PATH, MIGRATION_ID, BINDING_STAGE_SILVER, NOTEBOOK_NAME, "succeeded")
    print(f"Applied: {MIGRATION_ID}")
